# Panel farm job-3 â€” render 293k champion-moment panels (Colab CPU)

Registered: `hypotheses.jsonl` â†’ `chart-native-encoder` (2026-10-23).
Moved off the VPS (claude 10-23: it competed with live EAs; Jay: use Colab Pro).

Renders all 293,727 entry-table moments as multi-TF panel JPEGs
(D1/H4/H1/M15 strips + H1 OBV/AO/AC, Jay's template) into Drive as
`panels_part_NNN.zip` checkpoints (5k panels each, ~60 zips total, ~18GB).

**Resume-safe**: re-run this notebook any number of times â€” finished parts
are skipped via `farm_done.txt` in the Drive folder. Suggest Colab Pro
**background execution** so it survives browser close. When the whole job
finishes, the final cell unassigns the runtime automatically.

In [ ]:
import os
PACK_URL = os.environ.get(
    "BRAIN_PACK_URL",
    "https://github.com/vicjayjay/brain-packs/releases/download/job1-v1/pack_job1.zip")
DRIVE_DIR = "/content/drive/MyDrive/brain_packs/panel_farm"
RAW = "https://raw.githubusercontent.com/vicjayjay/brain-packs/main/code/"
CODE_FILES = ["panel_farm_colab.py", "panel_situation_card.py",
              "render_jay_template_20261022.py", "tape_feel.py"]
print(PACK_URL)

In [ ]:
import subprocess, sys, zipfile, urllib.request
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "pandas", "pyarrow", "matplotlib"], check=True)
from google.colab import drive  # type: ignore
drive.mount("/content/drive")   # one consent

work = Path("/content/pack"); work.mkdir(exist_ok=True)
zip_local = work / "pack_job1.zip"
if not (work / "entries.parquet").exists():
    if not zip_local.exists():
        urllib.request.urlretrieve(PACK_URL, zip_local)
    with zipfile.ZipFile(zip_local) as z:
        z.extractall(work)
for f in CODE_FILES:
    dst = Path("/content") / f
    if dst.exists():
        dst.unlink()                       # always fresh: stale code caused the IndexError
    urllib.request.urlretrieve(RAW + f + "?v=2", dst)
sys.path.insert(0, "/content")
print("pack + code ready")

In [ ]:
import panel_farm_colab as farm
import sys
sys.argv = ["farm", "--pack-dir", "/content/pack", "--drive-dir", DRIVE_DIR]
farm.main()

In [ ]:
# Runs only when ALL panels are done (farm prints ALL DONE). Everything durable
# is already in Drive; unassign so the runtime stops billing immediately.
from google.colab import runtime  # type: ignore
runtime.unassign()